# 第 03 章：为研究任务接入可核验知识（离线工程实验）

**目标**：执行本章稳定公共契约并观察失败护栏。  
**环境与预计用时**：Python 3.12、offline profile，约 15–25 分钟。  
本 Notebook 由同名 Markdown 中带 `sync` 标识的实验代码生成；可复用业务逻辑始终从 `mini_deerflow` package 导入。

## 1. offline profile 初始化

显式选择离线模型档位，基础实验不得读取供应商 Key。

In [1]:
from mini_deerflow.config import ModelProfile, ModelSettings

lesson_settings = ModelSettings(profile=ModelProfile.OFFLINE)
assert lesson_settings.profile is ModelProfile.OFFLINE


## 2. 前置能力探针

验证当前 kernel 使用课程锁定的主版本，并能导入 Mini DeerFlow。

In [2]:
from importlib.metadata import version
import mini_deerflow

assert version('langchain').startswith('1.3.')
assert version('langgraph').startswith('1.2.')
assert mini_deerflow.__file__


## 3. 最小成功实验

以下单元来自 Markdown 的稳定 sync marker。

### 实验 `ch03-idempotent-index`

In [3]:
from mini_deerflow.knowledge import KnowledgeDocument, LocalKnowledgeIndex

index = LocalKnowledgeIndex()
document = KnowledgeDocument(
    id="durable-execution",
    text="Durable execution 通过 checkpoint 和 thread 恢复长任务。",
    source="official/persistence.md",
    metadata={"topic": "langgraph"},
)
first_report = index.upsert([document])
second_report = index.upsert([document])
assert (first_report.added, first_report.unchanged) == (1, 0)
assert (second_report.added, second_report.unchanged) == (0, 1)
assert len(index) == 1


### 实验 `ch03-search-with-source`

In [4]:
hits = index.search("checkpoint 如何恢复 durable execution", limit=1)
assert len(hits) == 1
assert hits[0].source == "official/persistence.md"
assert "thread" in hits[0].text
hits[0]


### 实验 `ch03-retriever-tool`

In [5]:
from mini_deerflow.tools import build_search_knowledge_tool

search_knowledge = build_search_knowledge_tool(index)
tool_result = search_knowledge.invoke({"query": "checkpoint", "limit": 1})
assert "official/persistence.md" in tool_result
assert "Durable execution" in tool_result
tool_result


### 实验 `ch03-vector-filter`

In [6]:
from mini_deerflow.knowledge.indexer import VectorKnowledgeIndex

vector_index = VectorKnowledgeIndex(embedding_size=64)
vector_documents = [
    KnowledgeDocument(
        id="persistence",
        text="checkpoint thread durable execution recovery",
        source="official/persistence.md",
        metadata={"topic": "runtime"},
    ),
    KnowledgeDocument(
        id="structured-output",
        text="structured output schema validation",
        source="official/structured-output.md",
        metadata={"topic": "model"},
    ),
]
vector_index.upsert(vector_documents)
filtered_hits = vector_index.search(
    "checkpoint thread durable execution recovery",
    limit=2,
    metadata_filter={"topic": "runtime"},
)
assert [hit.id for hit in filtered_hits] == ["persistence"]


### 实验 `ch03-recall-evaluation`

In [7]:
from mini_deerflow.knowledge.evaluation import RetrievalCase, recall_at_k

retrieval_cases = [
    RetrievalCase(
        query="checkpoint thread durable execution recovery",
        expected_ids={"persistence"},
    ),
    RetrievalCase(
        query="structured output schema validation",
        expected_ids={"structured-output"},
    ),
]
assert recall_at_k(vector_index, retrieval_cases, k=1) == 1.0


## 4. 状态/事件观察

观察消息、结构化对象、检索命中或 v2 event；不要只看最终自然语言。

## 5. 失败实验

失败必须被捕获并断言，证明护栏真的阻止了错误路径。

### 实验 `ch03-empty-retrieval-failure`

In [8]:
empty_hits = index.search("量子引力弦理论", limit=1)
assert empty_hits == []


## 6. Mini DeerFlow 工程调用

以上实验只从 `mini_deerflow` 导入公共接口；Notebook 不复制 Agent、Tool 或 Schema 实现。

## 7. 分层练习

完成同名 Markdown 的练习 A（单点修改）、B（边界判断）、C（项目扩展）和延迟回忆题。先自行作答，再运行对应 pytest 获取即时反馈。

## 8. 自动验收摘要

在项目根目录运行 `make test`。本 Notebook 的所有代码单元必须有执行计数、不得保存 error output，教程验证结果不得出现本章 drift。

## 9. 清理临时资源

当前实验使用内存对象与 `TemporaryDirectory`，退出上下文后自动清理；不要把 API Key、向量库或临时产物写回仓库。